# 04 · Statistical analysis and independent cross-validation

The report's numbers were produced by a Node.js analysis (`experiments/harness/analyze.mjs`, with a hand-written statistics module). This notebook **recomputes them independently** with SciPy and Statsmodels and checks that both implementations agree. Deterministic statistics must match to numerical precision; bootstrap intervals use different random number generators, so they are compared with a tolerance.

In [1]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW, PROC, RES = ROOT / "data/raw", ROOT / "data/processed", ROOT / "experiments/results"
LOGS, CONFIG = ROOT / "experiments/logs", ROOT / "experiments/configs/experiments.json"

def load_events(run):
    """One row per logged event (op = one transaction attempt, trial = one trial)."""
    return pd.read_json(RAW / run / "events.jsonl", lines=True)

ORDER = ["naive", "conditional_update", "pessimistic_sorted", "pessimistic_unordered",
         "optimistic_version", "serializable", "advisory_lock"]
COLORS = dict(zip(ORDER, ["#d92d20", "#F84464", "#2e90fa", "#7a5af8", "#12b76a", "#f79009", "#667085"]))
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.width", 140)
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.proportion import proportion_confint

In [2]:
neon, live = load_events("main-neon"), load_events("main-live")
ops = neon[(neon.type == "op") & (~neon.warmup.fillna(False).astype(bool))]
T = neon[(neon.type == "trial") & (~neon.warmup.fillna(False).astype(bool))]
js = json.loads((RES / "results.json").read_text())
e1_js = pd.read_csv(PROC / "e1_hot_seat.csv")
e3_js = pd.read_csv(PROC / "e3_low_contention.csv")
pw_js = pd.read_csv(PROC / "e3_pairwise_tests.csv")
checks = []
def check(name, py, node, rtol=1e-6, atol=1e-9):
    ok = bool(np.isclose(py, node, rtol=rtol, atol=atol))
    checks.append((name, py, node, ok))

C:\Users\user\AppData\Local\Temp\ipykernel_5808\1970957938.py:2: Pandas4Warning: 'bool' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  ops = neon[(neon.type == "op") & (~neon.warmup.fillna(False).astype(bool))]
C:\Users\user\AppData\Local\Temp\ipykernel_5808\1970957938.py:3: Pandas4Warning: 'bool' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  T = neon[(neon.type == "trial") & (~neon.warmup.fillna(False).astype(bool))]


## Proportions: Wilson intervals and Fisher's exact test

In [3]:
for _, r in e1_js.iterrows():
    lo, hi = proportion_confint(r.trials_with_double_booking, r.trials, method="wilson")
    check(f"Wilson low  {r.strategy} c={r.concurrency}", lo, r.p_ci_low, rtol=1e-3, atol=1e-4)
    check(f"Wilson high {r.strategy} c={r.concurrency}", hi, r.p_ci_high, rtol=1e-3, atol=1e-4)
h1 = js["hypotheses"]["H1"]
_, p = stats.fisher_exact([[h1["naive"]["withDoubleBooking"], h1["naive"]["trials"] - h1["naive"]["withDoubleBooking"]],
                           [h1["conditional_update"]["withDoubleBooking"], h1["conditional_update"]["trials"] - h1["conditional_update"]["withDoubleBooking"]]])
check("Fisher p (H1)", p, h1["fisherExactP"], rtol=1e-6)
h5 = js["hypotheses"]["H5"]
_, p5 = stats.fisher_exact([[h5["neon"]["withDoubleBooking"], h5["neon"]["trials"] - h5["neon"]["withDoubleBooking"]],
                            [h5["pglite"]["withDoubleBooking"], h5["pglite"]["trials"] - h5["pglite"]["withDoubleBooking"]]])
check("Fisher p (H5)", p5, h5["fisherExactP"], rtol=1e-6)

## Percentiles (type 7 / linear interpolation in both implementations)

In [4]:
for _, r in e1_js.iterrows():
    x = ops[(ops.exp == "E1") & (ops.strategy == r.strategy) & (ops.c == r.concurrency)].latencyMs
    check(f"p50 {r.strategy} c={r.concurrency}", np.percentile(x, 50), r.latency_p50_ms, rtol=1e-3)
    check(f"p95 {r.strategy} c={r.concurrency}", np.percentile(x, 95), r.latency_p95_ms, rtol=1e-3)

## Rank tests: Kruskal–Wallis, Mann–Whitney U (Holm-adjusted) and Cliff's δ

In [5]:
e3 = ops[(ops.exp == "E3") & (ops.outcome == "success")]
for t in js["hypotheses"]["H3"]["tests"]:
    c = t["concurrency"]
    groups = [g.latencyMs.values for _, g in e3[e3.c == c].groupby("strategy")]
    H, _ = stats.kruskal(*groups)
    check(f"Kruskal-Wallis H c={c}", H, t["kruskalWallis"]["H"], rtol=1e-3)
    ref = e3[(e3.c == c) & (e3.strategy == "conditional_update")].latencyMs.values
    ps = []
    for row in t["pairwise"]:
        s = row["comparison"].split(" vs ")[0]
        x = e3[(e3.c == c) & (e3.strategy == s)].latencyMs.values
        U, p = stats.mannwhitneyu(x, ref, alternative="two-sided", use_continuity=True, method="asymptotic")
        ps.append(p)
        check(f"Mann-Whitney U {s} c={c}", U, row["U"], rtol=1e-6)
        check(f"Cliff delta  {s} c={c}", 2 * U / (len(x) * len(ref)) - 1, row["cliffs_delta"], atol=1e-3)
        if row["p_raw"] > 1e-300:
            check(f"MW p (raw)   {s} c={c}", p, row["p_raw"], rtol=1e-3, atol=1e-12)
    for row, ph in zip(t["pairwise"], multipletests(ps, method="holm")[1]):
        if row["p_holm"] > 1e-300:
            check(f"Holm p       {row['comparison'].split(' vs ')[0]} c={c}", ph, row["p_holm"], rtol=1e-3, atol=1e-12)

## Bootstrap intervals (different RNGs — compared with a 5% tolerance on the interval bounds)

In [6]:
for _, r in e3_js.iterrows():
    tt = T[(T.exp == "E3") & (T.strategy == r.strategy) & (T.c == r.concurrency)]
    thr = (tt.successes / tt.makespanMs * 1000).values
    check(f"throughput mean {r.strategy} c={r.concurrency}", thr.mean(), r.throughput_tps_mean, rtol=1e-2, atol=0.01)
    bs = stats.bootstrap((thr,), np.mean, method="percentile", n_resamples=5000, random_state=7).confidence_interval
    check(f"throughput CI low  {r.strategy} c={r.concurrency}", bs.low, r.throughput_ci_low, rtol=0.05, atol=0.05)
    check(f"throughput CI high {r.strategy} c={r.concurrency}", bs.high, r.throughput_ci_high, rtol=0.05, atol=0.05)

## Live system (E4): H4 decision recomputed

In [7]:
reads = live[(live.type == "op") & (live.exp == "E4a") & (live.status == 200) & (~live["first"].fillna(False).astype(bool))]
rows = []
for ep, g in reads.groupby("endpoint"):
    x = g.latencyMs.values
    bs = stats.bootstrap((x,), lambda a, axis: np.percentile(a, 95, axis=axis), method="percentile", n_resamples=5000, random_state=3).confidence_interval
    rows.append((ep, len(x), np.median(x), np.percentile(x, 95), bs.low, bs.high, (x < 200).mean(), bs.high < 200))
h4 = pd.DataFrame(rows, columns=["endpoint", "n", "p50", "p95", "p95_ci_low", "p95_ci_high", "share_lt_200", "meets_target"]).round(2)
h4

C:\Users\user\AppData\Local\Temp\ipykernel_5808\963255168.py:1: Pandas4Warning: 'bool' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  reads = live[(live.type == "op") & (live.exp == "E4a") & (live.status == 200) & (~live["first"].fillna(False).astype(bool))]


,endpoint,n,p50,p95,p95_ci_low,p95_ci_high,share_lt_200,meets_target
0,health,117,118.45,150.25,139.39,153.29,0.98,True
1,movies,117,37.68,59.77,52.67,83.63,0.99,True
2,seats,117,122.69,199.90,145.05,417.20,0.95,False
3,showtimes,117,123.95,146.44,137.68,175.27,0.99,True
4,static,27,33.36,54.19,44.66,109.53,1.00,True


## Cross-validation result

In [8]:
cv = pd.DataFrame(checks, columns=["quantity", "python", "node", "agree"])
print(f"{cv.agree.sum()} of {len(cv)} quantities agree between the SciPy/Statsmodels and Node implementations")
cv[~cv.agree] if (~cv.agree).any() else cv.sample(10, random_state=0)

154 of 154 quantities agree between the SciPy/Statsmodels and Node implementations


,quantity,python,node,agree
26,Wilson low pessimistic_sorted c=8,0.000000e+00,0.000000e+00,True
89,MW p (raw) naive c=1,4.444048e-07,4.444048e-07,True
151,throughput mean advisory_lock c=16,2.825953e+00,2.830000e+00,True
109,Cliff delta naive c=16,9.616840e-01,9.617000e-01,True
24,Wilson low pessimistic_sorted c=2,0.000000e+00,0.000000e+00,True
7,Wilson high conditional_update c=2,1.611252e-01,1.611000e-01,True
44,p50 advisory_lock c=2,5.959190e+02,5.959200e+02,True
107,Kruskal-Wallis H c=16,1.936154e+03,1.936150e+03,True
56,p50 naive c=2,6.507895e+02,6.507900e+02,True
121,throughput mean conditional_update c=1,2.714383e+00,2.710000e+00,True


In [9]:
assert cv.agree.all(), "implementations disagree — see table above"
print("All statistics independently reproduced.")

All statistics independently reproduced.
